In [1]:
%run Coding_Crypto_Lib.ipynb
%run GRS_classical_square_code_attack_Lib.ipynb
from itertools import chain

## Functions for the Attack

In [2]:
def find_cancellations(F, n, G_prime, cube_dim):
    F_list = F.list()[1:]
    cycles = []
    overlaps = {}
    for i in range(n):
        overlaps[i] = []
    visited = [False]*n
    
    for i in range(n):
        if visited[i]:
            continue
        w = [i]
        visited[i] = True
        break_while = False
        while True:
            curr = w[-1]
            if len(w) >= 2:
                prev = w[-2]
            else:
                prev = None
            for j in range(n):
                if j == curr or j == prev or (visited[j] and j != i):
                    continue
                count_gamma_valid = 0
                for elt in F_list:
                    N = identity_matrix(F, n)
                    N[curr,j] = elt
                    G_prime_new = G_prime * N
                    cube = cube_code(G_prime_new)
                    if cube.nrows() < cube_dim:
                        count_gamma_valid += 1
                        overlaps[curr].append((j,elt))
                        overlaps[j].append((curr, 1/elt))
                if count_gamma_valid > 2:
                    w.append(j)
                    visited[j] = True
                    cycles.append(w)
                    break_while = True
                    break
                elif count_gamma_valid == 1:
                    if j == i:
                        cycles.append(w)
                        break_while = True
                        break
                    else:
                        w.append(j)
                        visited[j] = True
                        break
            if break_while:
                break
    return overlaps, cycles


def filter_4_cycles(cycles):
    cycles_new = []
    four_cycles = []
    for cycle in cycles:
        if len(cycle) != 2:
            cycles_new.append(cycle)
        else:
            four_cycles.append(cycle)
    return cycles_new, four_cycles

def create_M_new(overlaps, cycles, F, n):
    # assuming no 4-cycles.
    M_new = zero_matrix(F, n, n)

    ind = 0
    for j in range(len(cycles)):
        for i in range(len(cycles[j])-1):
            curr = cycles[j][i]
            M_new[i + ind, curr] = 1
            for elt in overlaps[curr]:
                nex, mult = elt
                if nex == cycles[j][i+1]:
                    M_new[i + ind, nex] = -mult
        
        # go back to start
        i = len(cycles[j]) - 1
        curr = cycles[j][i]
        M_new[i + ind, curr] = 1
        for elt in overlaps[curr]:
            nex, mult = elt
            if nex == cycles[j][0]:
                M_new[i + ind, nex] = - mult
        ind += len(cycles[j])
    return M_new

def make_perm(cycles_new, four_cycles):
    perm_cycles_new = list(chain.from_iterable(cycles_new))
    perm_four_cycles = list(chain.from_iterable(four_cycles))
    perm_0 = perm_cycles_new + perm_four_cycles
    perm_list = [elt + 1 for elt in perm_0]
    return Permutation(perm_list).to_matrix()

def make_mask_identity_for_4_cycles(M_new_perm, num_four_cycles):
    n = M_new_perm.nrows()
    dim_second_part = 2 * num_four_cycles
    return block_diagonal_matrix(M_new_perm[:n-dim_second_part,:n-dim_second_part], identity_matrix(dim_second_part), subdivide=False)

def four_cycle_attack(F, G_square_dist, num_four_cycles, M_new_perm):
    F_list = F.list()[1:]
    n = G_square_dist.ncols()
    G_square_dist_now = G_square_dist
    M_new_perm_copy = copy(M_new_perm)
    for i in range(num_four_cycles, 0, -1):
        # One could make the square code smaller after iterations in this loop. I have not done so for now.
        square_dim = square_code_generator_full_rank(G_square_dist_now).nrows()
        multiplier = identity_matrix(F, n)
        break_loop = False
        for elt1 in F_list:
            if break_loop:
                break
            for elt2 in F_list:
                if break_loop:
                    break
                multiplier[n-2*i, n-2*i+1] = elt1
                multiplier[n-2*i+1, n-2*i] = elt2
                if square_code_generator_full_rank(G_square_dist_now * multiplier).nrows() < square_dim:
                    if elt1 != 1/elt2:
                        M_new_perm_copy[n-2*i,n-2*i] = 1
                        M_new_perm_copy[n-2*i+1,n-2*i+1] = 1
                        M_new_perm_copy[n-2*i,n-2*i+1] = - elt1
                        M_new_perm_copy[n-2*i+1,n-2*i] = - elt2
                        break_loop = True
    return M_new_perm_copy

def attack_weight_two_masked(G_prime):
    # returns the dual of the original GRS code together with the mask M.
    F = G_prime.base_ring()
    n = G_prime.ncols()
    H_prime = LinearCode(G_prime).parity_check_matrix()
    cube_dim = cube_code(H_prime).nrows() + 1

    # Find overlaps, cycles and the permutation that gives a block diagonal matrix.
    overlaps, cycles = find_cancellations(F, n, H_prime, cube_dim)
    cycles_new, four_cycles = filter_4_cycles(cycles)
    perm_block = make_perm(cycles_new, four_cycles)
    num_four_cycles = len(four_cycles)

    # find the entries of M that are not in a 4-cycle
    M_new = create_M_new(overlaps, cycles_new, F, n)
    M_new_perm = M_new * perm_block

    
    M_new_perm_id = make_mask_identity_for_4_cycles(M_new_perm, num_four_cycles)
    square_dist_code = H_prime * perm_block * (M_new_perm_id^(-1))
    M_new_perm_final = four_cycle_attack(F, square_dist_code, num_four_cycles, M_new_perm)
    code_info = attack(H_prime * perm_block * M_new_perm_final^(-1))
    correct_mask = M_new_perm_final * (perm_block.T)
    return code_info[0], correct_mask